# Dimensionality reduction 1

Practice applying and using **dimensionality reduction** for analysing datasets with Principal Component Analysis (`PCA` & `Kernel PCA`), but continue with the more powerful t-SNE and UMAP algorithms.

---
### Data

The dataset is a photometric catalogue of galaxies. These galaxies were found in the 2-square degree field on the sky called COSMOS by space- and ground-based telescopes.

The radiation flux (energy per second) of each galaxy is measured in 8 bands (i.e. wavelengths of light) that span the spectrum from <span style="color:blue;">blue</span> to <span style="color:rgb(192,4,1,1);">infrared</span>: `u, r, z++, yHSC, H, Ks, SPLASH1, SPLASH2`. The fluxes are not corrected for any effects, such as distance to a galaxy, therefore there is a systematic effect in their measurements (called redshift).

So, in addition to its photometry each galaxy has its observed bias and physical properties:
* `redshift`$^1$ - systematic bias in flux measurements.
* `log_mass` - stellar mass in units of $log_{10}$ (inferred from a combination of fluxes and redshifts).
* `log_sfr` - rate of star formation in units of $log_{10}$ (inferred from a combination of fluxes and redshifts).
* `is_star_forming` - classification, based on galaxy colours (inferred from a combinations of fluxes and redshifts).


$^1$ redshift is the reddening of light that is proportianal to the velocity of an object receding away. On the sky, object velocities are proportional to their distances from us ([find out more](https://www.anisotropela.dk/encyclo/redshift.html)).

---
* Authors:  Vadim Rusakov, Charles Steinhardt, Troels Petersen
* Email:  vadim.rusakov@nbi.ku.dk, petersen@nbi.dk
* Date:   5th of May 2025

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import umap
from matplotlib.colors import LogNorm
from sklearn.decomposition import PCA, KernelPCA
from sklearn.manifold import TSNE
import seaborn as sns
from sklearn import preprocessing

Load in the data:

In [ ]:
file = "datasets/cosmos2015.csv"
df = pd.read_csv(file, index_col=False)

df.head()
df.describe()

Select a random sub-sample of the dataset. `PCA` does computations linearly, therefore it's quick and you can choose the whole dataset if you wish.

In [ ]:
# # select a random sub-sample of the dataset
# n = 10000
# idxs = np.arange(df.shape[0])
# idxs_rand = np.random.choice(idxs, size=n)
# df_cut = df.iloc[idxs_rand] # dataframe
# X = df.iloc[idxs_rand].values # array

# flux_cols = list(df.columns[4:]) # flux column names
# flux_idxs = np.argwhere(np.isin(df.columns, flux_cols)).flatten() # flux column indices

from sklearn.model_selection import train_test_split

df_cut, _ = train_test_split(df, train_size=10000, random_state=42)

flux_cols = list(df.columns[4:])
flux_idxs = df.columns.get_indexer(flux_cols)

X = df_cut#.values

X_flux = X.iloc[:,flux_idxs] #[:,flux_idxs]
y_flux = df_cut.is_star_forming

params_cols = list(df.columns[:3])
params_idxs = df.columns.get_indexer(params_cols)
X_params = X.iloc[:,params_idxs]


# sns.pairplot(df_cut, vars=flux_cols, hue='is_star_forming', corner=True);

## Principal Component Analysis (PCA)

Now take the galaxy data (fluxes) and find out whether you can reduce it to a couple of meaningful principal components using `PCA`. By meaningful, we are interested in the method that is capable of separating galaxies into `star forming` or `dead`.

Use the following parameters: `n_components=2`. The user interface of the PCA in sklearn is the same as for all other similar classes (see PCA [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)).

You can access training data (only fluxes columns) as `X[:, flux_idxs]`.

In [ ]:
# a quick function for plotting our PCA components
def plot_pca(y_pcs, y, ax=0):
    
    if not ax:
        fig, ax = plt.subplots(1, figsize=(5, 5), dpi=100)
    #=== plot PCA results
    # fig, ax = plt.subplots(1, figsize=(5, 5), dpi=100)
    #ax.set_xlim(np.percentile(y_pcs[:,0], 99), np.percentile(y_pcs[:,0], 1))
    #ax.set_ylim(np.percentile(y_pcs[:,1], 99), np.percentile(y_pcs[:,1], 1))
    ax.set_xlabel('Component 1')
    ax.set_ylabel('Component 2')

    # locate the points of each type in the original data
    # and paint them over the transformed data
    # is_type1 = (y == 0)
    is_sf = (y == 1)
    ax.scatter(y_pcs[is_sf, 0], y_pcs[is_sf, 1], 
               c='b', s=0.05, label='star forming')
    ax.scatter(y_pcs[~is_sf, 0], y_pcs[~is_sf, 1], 
               c='r', s=0.05, label='dead',)
    
    
    ax.annotate("star forming", xy=(0.05, 0.9), xycoords="axes fraction", 
                color='b', fontsize=12)
    ax.annotate("dead", xy=(0.05, 0.86), xycoords="axes fraction", 
                color='r', fontsize=12)
    
    # ax.legend()
    # plt.show()
    
    if not ax:
        plt.show()
    
    return 

### Raw

In [ ]:
pca = PCA(n_components=2, svd_solver='full') # get a pca object of class PCA()

y_pcs = pca.fit_transform(X_flux) # train pca object on fluxes (raw observed data)

plot_pca(y_pcs, y_flux)

In [ ]:
fig, ax = plt.subplots(1, figsize=(12, 6), dpi=100)
ax.set_xlabel('log10( Variable values )')
ax.set_ylabel('Number')
ax.set_title('Raw data')
xbins = np.arange(-1, 3.5, 0.1)

feature_labels = X_flux.columns


for i in range(X_flux.shape[1]):
    data = X_flux.iloc[:, i]
    positive_data = data[data > 0]
    if len(positive_data) < len(data):
        print(f"Warning: {len(data) - len(positive_data)} non-positive values skipped for {feature_labels[i]}")
    log_data = np.log10(positive_data)
    ax.hist(log_data, bins=xbins, histtype='step', label=f'{i}. {feature_labels[i]}')

ax.legend(loc=1)
plt.show()

### Normalised

In [ ]:
transform = preprocessing.StandardScaler()
X_std = transform.fit_transform(X_flux)

pca = PCA(n_components=2, svd_solver='full') # get a pca object of class PCA()

y_pcs = pca.fit_transform(X_std) # train pca object on fluxes (raw observed data)

plot_pca(y_pcs, y_flux)

In [ ]:
fig, ax = plt.subplots(1, figsize=(12, 6), dpi=100)
ax.set_xlabel('log10( Variable values )')
ax.set_ylabel('Number')
ax.set_title('Raw data')
xbins = np.arange(-1, 3.5, 0.1)

feature_labels = X_flux.columns


for i in range(X_std.shape[1]):
    x_pos = X_std[:, i][X_std[:, i] > 0.0]
    ax.hist(np.log10(x_pos), bins=xbins, histtype='step', label=f'{i}. {feature_labels[i]}')

ax.legend(loc=1)
# ax.set_ylim(None, 1200)
plt.show()

### colormaps

* Make scatter plots coloured by different galaxy properties: `log_mass`, `log_sfr`, `redshift`. Is the low-dimensional representation meaningful in any one of the properties? Can you argue why?

Below is an example code for colouring the scatter by some property, eg., `log_mass`:

In [ ]:
import matplotlib.colors as mcolors


def plot_pcs_with_colormap(y_pcs, df_cut, features=('redshift', 'log_mass', 'log_sfr'),
                                     cmap=plt.cm.jet, percentile_clip=(1, 99), log_thresh=100):
    """
    Plot 2D PCA scatter plots for selected features using adaptive color normalizations (log or linear).

    Parameters:
        y_pcs (ndarray): PCA-transformed coordinates (N x 2).
        df_cut (DataFrame): DataFrame containing the features to color by.
        features (tuple): List of feature names in df_cut to color the plots.
        cmap (Colormap): Matplotlib colormap instance.
        percentile_clip (tuple): Percentiles for axis limits (to reduce outlier impact).
        log_thresh (float): If (max / min) > log_thresh and min > 0, use LogNorm for that feature.
    """
    fig, axes = plt.subplots(1, len(features), figsize=(6 * len(features), 5), dpi=100, constrained_layout=True)

    # PCA plot limits, consistent across subplots
    # xlim = (np.percentile(y_pcs[:, 0], percentile_clip[1]), np.percentile(y_pcs[:, 0], percentile_clip[0]))
    # ylim = (np.percentile(y_pcs[:, 1], percentile_clip[1]), np.percentile(y_pcs[:, 1], percentile_clip[0]))
    xlim = ylim = (None, None)

    for ax, feature in zip(axes, features):
        values = df_cut[feature].values
        vmin, vmax = np.nanmin(values), np.nanmax(values)

        # # Decide whether to use LogNorm
        # if vmin > 0 and vmax / vmin > log_thresh:
        #     norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)
        #     norm_type = "LogNorm"
        # else:
        #     norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
        #     norm_type = "Linear"
        norm = LogNorm() #mcolors.LogNorm(vmin=vmin, vmax=vmax)
        norm_type = "LogNorm"
        
        # Plot
        sc = ax.scatter(y_pcs[:, 0], y_pcs[:, 1], s=0.2, c=values, cmap=cmap, norm=norm)
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.set_xlabel('Component 1')
        ax.set_ylabel('Component 2')
        ax.set_title(f'{feature} ({norm_type})')

        # Colorbar
        cbar = plt.colorbar(sc, ax=ax)
        cbar.ax.set_ylabel(feature, rotation=270, labelpad=10)

    plt.show()



def plot_pcs_with_colormap_v1(y_pcs, df_cut, features=('log_mass', 'log_sfr', 'redshift'),
                                   cmap=plt.cm.jet, percentile_clip=(1, 99)):
    """
    Plot 2D PCA scatter plots for selected features using individual color normalizations and colorbars.

    Parameters:
        y_pcs (ndarray): PCA-transformed coordinates (N x 2).
        df_cut (DataFrame): DataFrame containing the features to color by.
        features (tuple): List of feature names in df_cut to color the plots.
        cmap (Colormap): Matplotlib colormap instance.
        percentile_clip (tuple): Percentiles for axis limits (to reduce outlier impact).
    """
    # Create the figure and subplots
    fig, axes = plt.subplots(1, len(features), figsize=(6 * len(features), 5), dpi=100, constrained_layout=True)

    # Set axis limits based on percentiles for each dimension
    xlim = (np.percentile(y_pcs[:, 0], percentile_clip[1]), np.percentile(y_pcs[:, 0], percentile_clip[0]))
    ylim = (np.percentile(y_pcs[:, 1], percentile_clip[1]), np.percentile(y_pcs[:, 1], percentile_clip[0]))

    # Loop through each feature to create a separate scatter plot and colorbar
    for ax, feature in zip(axes, features):
        values = df_cut[feature]

        # Normalize based on the feature's value range
        vmin, vmax = np.nanmin(values), np.nanmax(values)
        norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

        # Plot the scatter plot
        sc = ax.scatter(y_pcs[:, 0], y_pcs[:, 1], s=0.2, c=values, cmap=cmap, norm=norm)

        # Set axis limits and labels
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.set_xlabel('Component 1')
        ax.set_ylabel('Component 2')

        # Create the colorbar specific to the feature
        cbar = plt.colorbar(sc, ax=ax)
        cbar.ax.set_ylabel(feature, rotation=270, labelpad=10)
        ax.set_title(f'Color by {feature}')

    plt.show()

In [ ]:
plot_pcs_with_colormap(y_pcs, df_cut)

## Kernel PCA

For now, let us continue throwing these data at other algorithms to get some practice with them. `KernelPCA` is a variant of the PCA, which can use a range of kernels for non-linear operations. I.e., this extension gives flexibility in separating the data that are not linearly-separable.

Use the following parameters: `n_components=2`, `kernel='cosine'`. Make sure to try different kernels for reducing the dimensionality. See documentation for `KernelPCA` in **sklearn**.

For Kernel PCA see the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.KernelPCA.html#sklearn.decomposition.KernelPCA).

In [ ]:
kpca = KernelPCA(n_components=2, kernel='cosine')
y_pcs = kpca.fit_transform(X_std)

# plot new PCA results
plot_pca(y_pcs, y_flux)

* Again, make scatter plots coloured by different galaxy properties: `log_mass`, `log_sfr`, `redshift`. Is the low-dimensional representation more meaningful with this algorithm? Can you argue why?

In [ ]:
plot_pcs_with_colormap(y_pcs, df_cut)

In [ ]:
fig, ax = plt.subplots(1, figsize=(12, 6), dpi=100)
ax.set_xlabel('log10( Variable values )')
ax.set_ylabel('Number')
ax.set_title('Raw data')
xbins = np.arange(-1, 3.5, 0.1)

feature_labels = X_flux.columns


for i in range(X_std.shape[1]):
    x_pos = X_std[:, i][X_std[:, i] > 0.0]
    ax.hist(np.log10(x_pos), bins=xbins, histtype='step', label=f'{i}. {feature_labels[i]}')


ax.legend(loc=1)
# ax.set_ylim(None, 1200)
plt.show()

### t-SNE

Now, try to run `t-SNE` on the dataset (for examples or set-up see documentation for `t-SNE` on sklearn [website](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html)). Use `perplexity=50, method='barnes_hut', n_iter=1000, random_state=42, verbose=2` for now. In the next class we will put more emphasis on the importance of the optimal values for theses parameters.

* How well does `t-SNE` help to differentiate between two classes here?

* Do you get clusters of galaxies or a continuum?

* Which physical property is the most distinctly separated in the reduced space (again, use colouring of scatter to analyze this)?

In [ ]:
t_sne = TSNE(
    n_components=2,
    perplexity=50,
    method='barnes_hut', 
    max_iter=1000, 
    random_state=42, 
    verbose=2,
    # n_components=2,
    # learning_rate='auto',
    # init='random',
    )

y_tsne = t_sne.fit_transform(X_std)

plot_pca(y_tsne, y_flux)

In [ ]:
plot_pcs_with_colormap(y_tsne, df_cut)

### UMAP

Now try using `UMAP`. For documentation see the UMAP [webpage](https://umap-learn.readthedocs.io/en/latest/api.html). This has the same interface as the other embedding classes above. Use with `n_components=2, n_neighbors=50, random_state=42`. 

* Do you get something similar to `t-SNE`?

* How well can you map different properties in the reduced space?

* Do you get clusters or continuous distributions? Which physical property is the most strongly separable with `UMAP`?

In [ ]:
from umap import UMAP

In [ ]:
# Perform UMAP reduction to 2D
reducer = UMAP(
    n_components=2,
    n_neighbors=50,
    random_state=42,
    )
y = reducer.fit_transform(X_std)




In [ ]:
plot_pca(y, y_flux)
plot_pcs_with_colormap(y, df_cut)

---

## Exercise:

Analyze the galaxy catalogue applying dimensionality reduction to galaxy fluxes.

1. Apply `PCA` to fluxes. Can you find a base of principal compoenents that separates galaxies into star forming and dead? Does PCA give you a way to differentiate between various properties of galaxies?
2. Think about preprocessing the data, if you haven't yet, and see if you can find a more representative set of principal components.
3. Apply `Kernel PCA` afterwards. Does this give you a more meaningful vector space? If so, why?
4. Apply `t-SNE`. Does it give you a cleaner separation between objects with different properties?
5. Apply `UMAP`, for comparison.
6. Discuss what you see in the above cases and what information that gives you with your group.
7. Try to play around with the parameters of the algorithms, and get a feel for, how that changes the outcome.
8. Try to apply the t-SNE and UMAP algorithms to data with flaws in (i.e. NaN values and/or heavy outliers), and see how they respond. Can you make them good at detecting such flaws?

## Learning points:

0. Dimensionality reduction is an unsupervised learning method, which is quite useful, as it lets you reduce data dimensionality to (typically) 2D, which you can plot at look at.
1. The PCA method is "only linear" and a standard that is very simple, very fast, and which you should know.<br>
  If you believe there are (mostly) linear relations in the data, use PCA.
2. The kernal PCA is less used (due to tSNE and UMAP), and included partially for illustration.
3. tSNE and UMAP are more powerful than PCA, because they are non-linear. They are however slow.<br>
  Therefore, for large datasets, only apply these to a fraction of the data (e.g. 10000 random events).